In [1]:
import os
os.chdir(r'c:\Users\leopa\Documents\python codes\New folder\Information_retrieval\notebooks')
print(f"Current working directory: {os.getcwd()}")

Current working directory: c:\Users\leopa\Documents\python codes\New folder\Information_retrieval\notebooks


# Step 2 & 3: Evaluation Setup & Baseline BM25

In this notebook, we:
1. Split the data into 50% Train (to generate queries) and 50% Test (the searchable database).
2. Setup Level 1 and Level 2 evaluation functions leveraging metadata (ONLY for evaluation).
3. Run the BM25 probabilistic model as our baseline.
4. Compute the average Ranking Errors.

In [2]:
# Imports
import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi
import warnings
warnings.filterwarnings('ignore')

### 1. Load Data & Train/Test Split

In [3]:
# Load prepared textual data
df_reviews = pd.read_csv('prepared_reviews.csv')

# Load metadata for evaluation
df_places = pd.read_csv('../data/Tripadvisor.csv', low_memory=False)
eval_cols = ['id', 'typeR', 'activiteSubType', 'restaurantType', 'restaurantTypeCuisine', 'priceRange']
df_places = df_places[eval_cols].copy()

# Merge to have text + metadata in one dataframe for splitting
df_merged = pd.merge(df_reviews, df_places, left_on='idplace', right_on='id', how='inner')
print(f"Total valid places with text and metadata: {len(df_merged)}")

# Split: 50% Train (Queries) | 50% Test (Search Base)
np.random.seed(42)
shuffled_indices = np.random.permutation(len(df_merged))
split_idx = len(df_merged) // 2

train_df = df_merged.iloc[shuffled_indices[:split_idx]].copy()
test_df = df_merged.iloc[shuffled_indices[split_idx:]].copy().reset_index(drop=True)

print(f"Train setup (Queries): {len(train_df)} places")
print(f"Test setup (Corpus): {len(test_df)} places")

Total valid places with text and metadata: 1835
Train setup (Queries): 917 places
Test setup (Corpus): 918 places


### 2. Evaluation Logic

We need to find the rank $n$ of the first correct match, from a sorted array of recommendations. The error is $n-1$.
If no test place matches the requirement, the error is undefined (we yield `None` and ignore it in the average).

### 2. Evaluation Logic

We find the rank $n$ of the first correct match; error = $n-1 = \text{index}$. Undefined when no valid match exists in the test set.

**FIX**: `extract_subcategories` is now type-aware: it only compares attributes that are semantically meaningful for a given venue type (e.g., `priceRange` only for Hotels, `cuisine` only for Restaurants). This prevents spurious cross-type matches that inflated Level 2 scores.

In [4]:
def eval_level_1(query_typeR, sorted_test_typeR_list):
    """Level 1 error: rank of first match on typeR (H/R/A/AP). Error = rank index."""
    if pd.isna(query_typeR) or query_typeR not in sorted_test_typeR_list:
        return None
    for i, t in enumerate(sorted_test_typeR_list):
        if t == query_typeR:
            return i
    return None

def extract_subcategories(row):
    """
    Type-aware subcategory extraction (FIX: avoids spurious cross-type matches).

    Previously all subcategory columns were pooled into one flat set, which
    caused a hotel's priceRange to 'match' a restaurant's priceRange even though
    they are completely different venue types.  Now we only compare attributes
    that are semantically relevant for the place's own typeR.

    Per assignment spec:
      H  (Hotel)             -> priceRange
      R  (Restaurant)        -> restaurantType, restaurantTypeCuisine
      A / AP (Attraction)    -> activiteSubType
    """
    cats = set()
    type_r = str(row.get('typeR', '')).strip()
    if type_r == 'H':
        val = row.get('priceRange', '')
        if pd.notna(val) and str(val).strip():
            for p in str(val).split(','):
                cats.add(p.strip().lower())
    elif type_r == 'R':
        for col in ['restaurantType', 'restaurantTypeCuisine']:
            val = row.get(col, '')
            if pd.notna(val) and str(val).strip():
                for p in str(val).split(','):
                    cats.add(p.strip().lower())
    elif type_r in ('A', 'AP'):
        val = row.get('activiteSubType', '')
        if pd.notna(val) and str(val).strip():
            for p in str(val).split(','):
                cats.add(p.strip().lower())
    return cats

# Pre-calculate test subcategories
test_subcats_list = [extract_subcategories(row) for _, row in test_df.iterrows()]
test_df['subcategories'] = test_subcats_list

def eval_level_2(query_subcats, sorted_test_indices, test_subcats_list):
    """Level 2 error: rank of first test place sharing at least one subcategory."""
    if not query_subcats:
        return None
    for i, test_idx in enumerate(sorted_test_indices):
        if query_subcats.intersection(test_subcats_list[test_idx]):
            return i
    return None


### 3. BM25 Baseline Setup

In [5]:
print("Tokenizing test documents for BM25...")
corpus = test_df['top_100_words'].fillna('').astype(str).tolist()
tokenized_corpus = [doc.split(" ") for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)
print("BM25 Indexed")

Tokenizing test documents for BM25...
BM25 Indexed


### 4. Running the Benchmark

In [6]:
import time

lvl1_errors = []
lvl2_errors = []

start_time = time.time()
print(f"Evaluating {len(train_df)} queries...")

for idx, row in train_df.iterrows():
    query_text = str(row['top_100_words']).strip()
    if not query_text or query_text == 'nan':  # FIX: str(NaN)='nan' is truthy
        continue
        
    tokenized_query = query_text.split(" ")
    
    # Get fast scores array: index matches the index in test_df
    doc_scores = bm25.get_scores(tokenized_query)
    
    # Get argsort descending
    ranked_indices = np.argsort(doc_scores)[::-1]
    
    # Build Ranked list of typeR for Level 1
    test_typeR_list = test_df['typeR'].values[ranked_indices].tolist()
    
    # --- EVAL LEVEL 1 ---
    err_1 = eval_level_1(row['typeR'], test_typeR_list)
    if err_1 is not None:
        lvl1_errors.append(err_1)
        
    # --- EVAL LEVEL 2 ---
    query_subcats = extract_subcategories(row)
    err_2 = eval_level_2(query_subcats, ranked_indices, test_subcats_list)
    if err_2 is not None:
        lvl2_errors.append(err_2)

print(f"Evaluation done in {time.time() - start_time:.2f}s")
print("-" * 30)
print(f"BM25 Average Ranking Error Level 1: {np.mean(lvl1_errors):.2f} (Computed on {len(lvl1_errors)} valid queries)")
print(f"BM25 Average Ranking Error Level 2: {np.mean(lvl2_errors):.2f} (Computed on {len(lvl2_errors)} valid queries)")

Evaluating 917 queries...


Evaluation done in 9.03s
------------------------------
BM25 Average Ranking Error Level 1: 0.64 (Computed on 917 valid queries)
BM25 Average Ranking Error Level 2: 4.41 (Computed on 414 valid queries)
